# v16.8 — gerbang kualitas sinyal (SQI templat QRS) untuk S1

Rancangan dikunci sebelum SQI/SNR dihitung: [RANCANGAN.md](RANCANGAN.md) (+ amandemen A; sha256 di `../logs/v16.8_praregistrasi.log`).
Logika di `gerbang.py`. Tahap A memilih ambang q di data pengembangan; tahap B memvalidasi sekali di afpdb + derau dan Icentia11k.

In [1]:
import json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import gerbang as G
C, K = G.C, G.K
import sys; sys.path[:0] = ["../v11_model_tim"]
from bersama import evaluasi as Ev
t0 = time.time()
R = {}
G.jalankan_bising()                     # dilewati bila cache lengkap
D = G.data_dev()
print({k: int(v["rec"].size) for k, v in D.items()})

  SQI LTAFDB: 84 rekaman


    16/84


    32/84


    48/84


    64/84


    80/84


... (12 outputs left out)

    2/12
    4/12


    6/12
    8/12
    10/12


    12/12
  SQI mitdb: 2 rekaman


    1/2
    2/2
{'LTAFDB': 1272328, 'MIT-BIH': 14152, 'NSRDB': 294162, 'nstdb': 4156, 'mitdb': 720}


## 1. Tahap A: grid q di data pengembangan

In [2]:
SNR_DEV = ("12", "06", "00", "_6")
def nst(d, snrs):
    m = np.isin(d["grup"], snrs)
    return G.subset(d, m)
rows = []
for q in (None,) + G.GRID:
    lt, mi, ns = G.nilai(D["LTAFDB"], q), G.nilai(D["MIT-BIH"], q), G.nilai(D["NSRDB"], q)
    nb = G.nilai(nst(D["nstdb"], SNR_DEV), q)
    per = {s: G.nilai(nst(D["nstdb"], (s,)), q)["palsu_per_jam"] for s in ("24", "18") + SNR_DEV}
    rows.append(dict(q=q, se_ltafdb=lt["se_episode_30"], se_mit=mi["se_episode_30"], palsu_nsrdb=ns["palsu_per_jam"],
                     palsu_nstdb_berat=nb["palsu_per_jam"], **{f"palsu_nstdb_{s}": v for s, v in per.items()},
                     gerbang_ltafdb=lt["frac_gerbang"], gerbang_nsrdb=ns["frac_gerbang"], gerbang_nstdb_berat=nb["frac_gerbang"]))
tabA = pd.DataFrame(rows); display(tabA.round(4))
b0 = rows[0]
syarat = [r for r in rows[1:] if b0["se_ltafdb"] - r["se_ltafdb"] <= 0.01 and b0["se_mit"] - r["se_mit"] <= 0.01]
if syarat:
    top = min(r["palsu_nstdb_berat"] for r in syarat)
    q_pilih = min(r["q"] for r in syarat if r["palsu_nstdb_berat"] <= top + 0.05)
else:
    q_pilih = None
R["tahapA"] = dict(grid=rows, memenuhi_syarat=[r["q"] for r in syarat], terpilih=q_pilih)
print("q yang memenuhi syarat Se:", [r["q"] for r in syarat], "→ TERPILIH:", q_pilih)

,q,se_ltafdb,se_mit,palsu_nsrdb,palsu_nstdb_berat,palsu_nstdb_24,palsu_nstdb_18,palsu_nstdb_12,palsu_nstdb_06,palsu_nstdb_00,palsu_nstdb__6,gerbang_ltafdb,gerbang_nsrdb,gerbang_nstdb_berat
0,NaN,0.9411,0.9773,1.2336,5.3019,0.0,0.0,1.0056,7.4227,7.6248,5.4545,0.0000,0.0000,0.0000
1,0.5,0.9412,0.9773,1.2338,5.3019,0.0,0.0,1.0056,7.4227,7.6248,5.4545,0.0001,0.0001,0.0000
2,0.6,0.9428,0.9773,1.0971,5.3097,0.0,0.0,1.0056,7.4446,7.6364,5.4628,0.0008,0.0027,0.0015
3,0.7,0.9440,0.9773,0.5394,5.3751,0.0,0.0,1.0112,7.5112,7.7538,5.5728,0.0033,0.0199,0.0136
4,0.8,0.9533,0.9773,0.2039,4.2122,0.0,0.0,0.0000,4.5212,7.0936,5.8632,0.0103,0.0517,0.0560
5,0.9,0.9764,0.9815,0.1811,0.0000,0.0,0.0,0.0000,0.0000,0.0000,0.0000,0.0507,0.0947,0.3292


q yang memenuhi syarat Se: [0.5, 0.6, 0.7, 0.8, 0.9] → TERPILIH: 0.9


## 2. Tahap B: validasi (afpdb + derau, Icentia)

In [3]:
B = G.pasang_sqi("afpdb_bising", G.muat_bising())
A0, _ = C.muat("afpdb"); A0 = G.pasang_sqi("afpdb", A0)
ic, _ = K.muat("Icentia"); ic = G.pasang_sqi("Icentia", ic)
val = {}
def banding(d, nama):
    out = {}
    for s, k in (("S1", "d1"), ("R", "v31")):
        m0, m1 = G.nilai(d, None, k), G.nilai(d, q_pilih, k)
        out[s] = {kk: m0[kk] for kk in ("palsu_per_jam", "se_episode_30", "jam")}
        out[s + "G"] = {kk: m1[kk] for kk in ("palsu_per_jam", "se_episode_30", "jam", "frac_gerbang")}
        out[s + "_unit"] = (m0["palsu_unit"], m1["palsu_unit"], m0["se_unit"], m1["se_unit"])
    return out
val["afpdb bersih"] = banding(A0, "afpdb")
for dz in G.DERAU:
    for s in G.SNR_VAL:
        val[f"afpdb + {dz} {s} dB"] = banding(G.subset(B, (B["derau"] == dz) & (B["snr"] == s)), dz)
val["Icentia AF"] = banding(K.bagian(ic, ic["grup"] == "AF"), "ic")
val["Icentia non-AF"] = banding(K.bagian(ic, ic["grup"] == "non-AF"), "ic")
R["validasi"] = {k: {s: v[s] for s in ("S1", "S1G", "R", "RG")} for k, v in val.items()}
display(pd.DataFrame({k: {f"{s} {kk}": round(vv, 3) for s in ("S1", "S1G", "R", "RG") for kk, vv in v[s].items() if kk != "jam"}
                      for k, v in R["validasi"].items()}).T)

  SQI afpdb_bising: 600 rekaman


    120/600


    240/600


    360/600


    480/600


    600/600


... (9 outputs left out)

    360/600


    480/600


    600/600


,S1 palsu_per_jam,S1 se_episode_30,S1G palsu_per_jam,S1G se_episode_30,S1G frac_gerbang,R palsu_per_jam,R se_episode_30,RG palsu_per_jam,RG se_episode_30,RG frac_gerbang
afpdb bersih,0.893,NaN,0.876,NaN,0.049,2.213,NaN,1.431,NaN,0.049
afpdb + ma 12 dB,1.188,NaN,1.230,NaN,0.067,2.356,NaN,1.692,NaN,0.067
afpdb + ma 6 dB,1.360,NaN,1.341,NaN,0.095,2.449,NaN,1.757,NaN,0.095
afpdb + ma 0 dB,1.924,NaN,1.911,NaN,0.190,2.632,NaN,1.229,NaN,0.190
afpdb + em 12 dB,1.209,NaN,0.874,NaN,0.085,2.767,NaN,1.590,NaN,0.085
afpdb + em 6 dB,2.183,NaN,1.337,NaN,0.145,3.139,NaN,1.605,NaN,0.145
afpdb + em 0 dB,6.047,NaN,3.108,NaN,0.354,5.407,NaN,1.093,NaN,0.354
Icentia AF,0.029,0.94,0.028,0.931,0.112,0.395,0.899,0.317,0.88,0.112
Icentia non-AF,0.496,NaN,0.461,NaN,0.075,2.332,NaN,2.132,NaN,0.075


## 3. Kriteria V1–V3 (dikunci)

In [4]:
v1 = val["afpdb + ma 6 dB"]
h = G.boot2(v1["S1_unit"][1], v1["S1_unit"][0])
k = dict(q=q_pilih)
k["V1_palsu"] = bool(R["validasi"]["afpdb + ma 6 dB"]["S1G"]["palsu_per_jam"] <= 1.0)
k["V1_lebih_rendah"] = bool(h["p_kurang"] >= 0.95)
a = R["validasi"]["Icentia AF"]
k["V2"] = bool(a["S1G"]["se_episode_30"] >= 0.90 and a["S1"]["se_episode_30"] - a["S1G"]["se_episode_30"] <= 0.02)
k["V3"] = bool(R["validasi"]["Icentia non-AF"]["S1G"]["palsu_per_jam"] <= 1.0)
k["S1G_menggantikan_S1"] = bool(q_pilih is not None and all(k[x] for x in ("V1_palsu", "V1_lebih_rendah", "V2", "V3")))
R["V1_boot"] = h; R["keputusan"] = k
print("V1 bootstrap (S1G − S1, afpdb + ma 6 dB):", h); print(json.dumps(k))

V1 bootstrap (S1G − S1, afpdb + ma 6 dB): {'delta': -0.019392143218710745, 'ci': [-0.13902938472760856, 0.10430280098769777], 'p_kurang': 0.6465, 'p_lebih': 0.3535, 'ci_a': [0.2637797349638389, 2.717119216703126]}
{"q": 0.9, "V1_palsu": false, "V1_lebih_rendah": false, "V2": true, "V3": true, "S1G_menggantikan_S1": false}


## 4. Kesimpulan

In [5]:
v = R["validasi"]
print(f"q terpilih: {q_pilih}")
for key in v:
    print(f"{key}: S1 palsu/jam {v[key]['S1']['palsu_per_jam']:.2f} → S1G {v[key]['S1G']['palsu_per_jam']:.2f} (gerbang {v[key]['S1G']['frac_gerbang']*100:.1f}% window); "
          f"Se {v[key]['S1']['se_episode_30']:.3f} → {v[key]['S1G']['se_episode_30']:.3f}")
print("Keputusan:", "S1G MENGGANTIKAN S1" if k["S1G_menggantikan_S1"] else "S1 tetap (kriteria gerbang tidak semua lolos)")
R["lingkungan"] = Ev._lingkungan(); R["durasi_menit"] = (time.time() - t0) / 60
G.simpan_json(R, G.CACHE / "ringkasan.json"); print("tersimpan")

q terpilih: 0.9
afpdb bersih: S1 palsu/jam 0.89 → S1G 0.88 (gerbang 4.9% window); Se nan → nan
afpdb + ma 12 dB: S1 palsu/jam 1.19 → S1G 1.23 (gerbang 6.7% window); Se nan → nan
afpdb + ma 6 dB: S1 palsu/jam 1.36 → S1G 1.34 (gerbang 9.5% window); Se nan → nan
afpdb + ma 0 dB: S1 palsu/jam 1.92 → S1G 1.91 (gerbang 19.0% window); Se nan → nan
afpdb + em 12 dB: S1 palsu/jam 1.21 → S1G 0.87 (gerbang 8.5% window); Se nan → nan
afpdb + em 6 dB: S1 palsu/jam 2.18 → S1G 1.34 (gerbang 14.5% window); Se nan → nan
afpdb + em 0 dB: S1 palsu/jam 6.05 → S1G 3.11 (gerbang 35.4% window); Se nan → nan
Icentia AF: S1 palsu/jam 0.03 → S1G 0.03 (gerbang 11.2% window); Se 0.940 → 0.931
Icentia non-AF: S1 palsu/jam 0.50 → S1G 0.46 (gerbang 7.5% window); Se nan → nan
Keputusan: S1 tetap (kriteria gerbang tidak semua lolos)
tersimpan
